In [46]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [47]:
from data_curation import DataLoader, SafeGroupKFold
from i3l_statistics import Statistics
from i3l_ml import ML
from enums import *
import json
import pandas as pd
from sklearn.model_selection import GroupKFold

# Load data and separate in train, test and external set

In [48]:
dl = DataLoader()
ml = ML()
stats = Statistics()

In [49]:
modes = [
    Mode.RWD, 
    # Mode.DP, 
    Mode.FMRAD, 
    # Mode.PYRAD, 
    # Mode.GEN
]

dataset = dl.create_dataset(
    modes=modes, 
    outcome='OS MONTHS',
    cohort=23,
    subanalysis=Subanalysis.CLASSIC
)

Selecting FM-RAD features with LASSO...


In [27]:
with open('split.json', 'r') as f:
    split = json.load(f)

train_set = dataset[dataset['Subject'].isin(split['TRAIN_SET'])].set_index('Subject')
test_set = dataset[dataset['Subject'].isin(split['TEST_SET'])].set_index('Subject')
ext_set = dataset[dataset['Subject'].str.startswith('UOC')].set_index('Subject')

In [37]:
X_train, y_train = train_set.drop(columns=['OS MONTHS', 'DEATH EVENT']), train_set[['OS MONTHS', 'DEATH EVENT']]
X_test, y_test = test_set.drop(columns=['OS MONTHS', 'DEATH EVENT']), test_set[['OS MONTHS', 'DEATH EVENT']]
X_ext, y_ext = ext_set.drop(columns=['OS MONTHS', 'DEATH EVENT']), ext_set[['OS MONTHS', 'DEATH EVENT']]

In [38]:
y_train = y_train.rename(columns={'OS MONTHS': 'TIME', 'DEATH EVENT': 'EVENT'})
y_test = y_test.rename(columns={'OS MONTHS': 'TIME', 'DEATH EVENT': 'EVENT'})
y_ext = y_ext.rename(columns={'OS MONTHS': 'TIME', 'DEATH EVENT': 'EVENT'})

In [41]:
with open('submodel_features.json', 'r') as f:
    submodel_features = json.load(f)
    submodel_features = [f for f in submodel_features if f in X_train.columns]

X_train = X_train.drop(columns=submodel_features)
X_test = X_test.drop(columns=submodel_features)
X_ext = X_ext.drop(columns=submodel_features)

In [43]:
X_train_imputed, imputer = dl.impute_df(X_train)
X_test_imputed, imputer = dl.impute_df(X_test, imputer=imputer)
X_ext_imputed, imputer = dl.impute_df(X_ext, imputer=imputer)

In [44]:
X_train_scaled, scaler, to_standard_normalize, to_log_normalize = dl.normalize(X_train_imputed)
X_test_scaled, _, _, _ = dl.normalize(X_test_imputed, scaler=scaler, to_standard_normalize=to_standard_normalize, to_log_normalize=to_log_normalize)
X_ext_scaled, _, _, _ = dl.normalize(X_ext_imputed, scaler=scaler, to_standard_normalize=to_standard_normalize, to_log_normalize=to_log_normalize)

6 features log normalized
11 features standardized
6 features log normalized
11 features standardized
6 features log normalized
11 features standardized


In [ ]:
train_set = 

# Train and evaluate

In [16]:
train_folds = dl.get_loco_folds(pd.Series(train_set.index))

cv = SafeGroupKFold(n_splits=len(train_folds.unique()))

def get_split():
    return cv.split(X_train_scaled, y_train, groups=train_folds)

In [21]:
model = ml.train_survival_model(
    dataset=train_set,
    model_name=Model.COX,
    cv=get_split,
    select_features=True
)
selected_features = model.feature_names_in_

UnboundLocalError: cannot access local variable 'X' where it is not associated with a value